# PyStrata workflow

Import the package directly from this repository's `src` directory.

In [1]:
import sys
from pathlib import Path

# Find the repository root whether Jupyter starts in the root or tests/.
repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "pystrata").is_dir()
)

src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import pystrata
from pystrata.output import KappaOutput
import pykooh

print(f"Using pystrata from: {Path(pystrata.__file__).resolve()}")

Using pystrata from: D:\Github\pystrata\src\pystrata\__init__.py


In [2]:
import numpy as np
import pyrvt
import pandas as pd

In [3]:
Site_Profile_df = pd.read_excel('data/Meloland_Soil_Profile.xlsx',
                               sheet_name = 'Scaled_Dmin_to_Kappa_0.036')

Layers = []
D_min = []
Vs = []
Thickness = []
Profile_Depth = []
mrd_strains = np.logspace(-6,0,num=20)
ModReduc_data = {}
Damping_data = {}
max_freqs = Site_Profile_df['Max Freq']
wave_fracs = Site_Profile_df['Wave Fraction']

In [ ]:

for i, (_, row) in enumerate(Site_Profile_df.iterrows()):

    soil_type = pystrata.site.DarendeliSoilType(unit_wt = row['Unit Weight (kN/m3)'],
                                                plas_index=row['PI'],
                                                ocr=1,
                                                stress_mean=row['Stress (kPa)'],
                                                strains = mrd_strains,
                                                damping_min = row['Scaled_D_min (%)']/100)
    
    Layers.append(pystrata.site.Layer(soil_type,row['Thickness (m)'],row['Velocity (m/s)']))

Layers.append(
            pystrata.site.Layer(
                pystrata.site.SoilType(
                    'Reference Rock',
                    25.9,
                    None,
                    0.01
                ),
                0,
                3500
            )
        )

In [5]:
Site_profile = pystrata.site.Profile(Layers)
discretized_Site_profile = Site_profile.auto_discretize(max_freq = max_freqs,wave_frac=wave_fracs)

In [6]:
def calc_kappa(freq,
               FAS,
               freq_range_for_kappa,
               ko_bandwidth = None,
               show_fit = False):
    
    if ko_bandwidth is not None:
        FAS_for_kappa = pykooh.smooth(
            freq_range_for_kappa,
            freq,
            FAS,
            bw = ko_bandwidth
        )
    else:
        FAS_for_kappa = np.interp(freq_range_for_kappa,freq,FAS)

    coeffs = np.polyfit(freq_range_for_kappa,np.log(FAS_for_kappa),1)
    slope, intercept = coeffs
    kappa = -slope / np.pi

    fit_line = np.exp(slope * freq_range_for_kappa + intercept)

    if show_fit:
        return fit_line
    else:
        return kappa

In [7]:
def Kappa_Correction(freq,FAS,Delta_kappa):

    FAS_adj = np.exp(-np.pi*Delta_kappa*freq)*FAS
    
    return FAS_adj

In [8]:
# Calculation Loop
outputs_freqs = np.logspace(np.log10(0.01),np.log10(50),1000)
RS_freqs = np.logspace(np.log10(0.05),np.log10(50),1000)

Kappa_freqs = outputs_freqs[
    (outputs_freqs >= 10) & (outputs_freqs <= 30)
]

output = pystrata.output.OutputCollection(
    [
        pystrata.output.FourierAmplitudeSpectrumOutput(
            outputs_freqs,
            pystrata.output.OutputLocation("outcrop", index=0),
            None
        ),
        pystrata.output.KappaOutput(
            Kappa_freqs,
            pystrata.output.OutputLocation("outcrop", index=0),
            None
        ),
        pystrata.output.KappaCorrectFourierAmplitudeSpectrumOutput(
            outputs_freqs,
            Kappa_freqs,
            0.039,
            pystrata.output.OutputLocation("outcrop", index=0),
            None
        )
    ] 
    )


motion = pystrata.motion.TimeSeriesMotion.load_at2_file(
    'data/NIS090.AT2'
)

eql_calc = pystrata.propagation.EquivalentLinearCalculator(strain_limit = 0.5)

In [9]:
p = discretized_Site_profile.copy()

eql_calc(motion, #type:ignore
        p,
        p.location("outcrop", index=-1))

In [10]:
output(eql_calc,
        name = f"test",)

In [11]:
FAS_df = output[0].to_dataframe()

FAS = FAS_df.iloc[:,0].values
freq = FAS_df.index.to_numpy()

In [12]:
kappa_verify = calc_kappa(freq,FAS,Kappa_freqs,None)
kappa = output[1].values

In [13]:
print(kappa_verify)
print(kappa)

0.48825322811228866
[0.48825323]


In [14]:
kappa_verify = calc_kappa(freq,FAS,Kappa_freqs,None)
kappa = output[1].values

In [15]:
delta_kappa = kappa - 0.039

In [16]:
fas_kappa_verify = Kappa_Correction(freq,FAS,delta_kappa)

fas_kappa = output[2].values

error = fas_kappa - fas_kappa_verify

In [17]:
print(error)

[3.99790670e-09 4.03515526e-09 4.07277467e-09 4.11076900e-09
 4.14914236e-09 4.18789894e-09 4.22704295e-09 4.26657866e-09
 4.30651041e-09 4.34684256e-09 4.38757957e-09 4.42872592e-09
 4.47028615e-09 4.51226487e-09 4.55466674e-09 4.59749648e-09
 4.64075888e-09 4.68445876e-09 4.72860103e-09 4.77319066e-09
 4.81823266e-09 4.86373213e-09 4.90969421e-09 4.95612414e-09
 5.00302718e-09 5.05040870e-09 5.09827411e-09 5.14662890e-09
 5.19547864e-09 5.24482894e-09 5.29468553e-09 5.34505417e-09
 5.39594071e-09 5.44735108e-09 5.49929128e-09 5.55176739e-09
 5.60478558e-09 5.65835207e-09 5.71247318e-09 5.76715533e-09
 5.82240499e-09 5.87822873e-09 5.93463321e-09 5.99162516e-09
 6.04921143e-09 6.10739892e-09 6.16619465e-09 6.22560572e-09
 6.28563933e-09 6.34630276e-09 6.40760341e-09 6.46954875e-09
 6.53214638e-09 6.59540397e-09 6.65932931e-09 6.72393030e-09
 6.78921492e-09 6.85519127e-09 6.92186757e-09 6.98925214e-09
 7.05735340e-09 7.12617989e-09 7.19574028e-09 7.26604334e-09
 7.33709796e-09 7.408913